# Schizophrenia Pathway Classifier — EDA

## Objective
In this notebook I will load the `GSE21138` dataset and inspect it, by first checking data quality and class balance to ensure the data is prepared enough to do accurate analysis and determine any potential feature engineering decisions. Then, after ensuring the data is good, conduct the actual exploratory analysis — seeing if the expression data separates by diagnosis or duration in reduced dimensions. Finally, concluding with a summary of EDA findings.

## Inputs
Dataset: `GSE21138` from "Gene Expression Profiles in BA46 of Subjects with Schizophrenia and Matched Controls".

Contains 30 subjects with schizophrenia and 29 age- and sex-matched controls, Ages (18-81 years), and three duration categories (short doi=<5 yrs; intermediate doi=7-18yrs; long doi=>28 yrs). 'Cont-7' was determined to be an outlier, and was removed from the publication analysis. 

Platform: GPL570 / Affymetrix Human Genome U133 Plus 2.0 Array

## 1.1 Setup & Imports

Import necessary libraries for data manipulation and plotting: numpy, pandas, matplotlib, seaborn, GEOparse. Additionally, set theme and figure size for plots. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse 

sns.set_theme()
plt.rcParams['figure.figsize'] = (10, 6)

## 1.2 Data Acquisition 
To access the GEO dataset I use `GEOparse.get_GEO` to save the GEO in `../data/raw`, and then confirm data was loaded properly by checking object type and sample count. 

In [2]:
gse = GEOparse.get_GEO(geo='GSE21138', destdir="../data/raw")

14-Aug-2026 12:18:01 DEBUG utils - Directory ../data/raw already exists. Skipping.
14-Aug-2026 12:18:01 INFO GEOparse - File already exist: using local version.
14-Aug-2026 12:18:01 INFO GEOparse - Parsing ../data/raw/GSE21138_family.soft.gz: 
14-Aug-2026 12:18:01 DEBUG GEOparse - DATABASE: GeoMiame
14-Aug-2026 12:18:01 DEBUG GEOparse - SERIES: GSE21138
14-Aug-2026 12:18:01 DEBUG GEOparse - PLATFORM: GPL570
/Users/joshuasim/Desktop/summer_projects/schizophrenia-pathway-classifier/venv/lib/python3.12/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
14-Aug-2026 12:18:02 DEBUG GEOparse - SAMPLE: GSM528831
14-Aug-2026 12:18:02 DEBUG GEOparse - SAMPLE: GSM528832
14-Aug-2026 12:18:02 DEBUG GEOparse - SAMPLE: GSM528833
14-Aug-2026 12:18:02 DEBUG GEOparse - SAMPLE: GSM528834
14-Aug-2026 12:18:02 DEBUG GEOparse - SAMPLE: GSM528835
14-Aug

In [33]:
print(type(gse))
len(gse.metadata['sample_id'])

<class 'GEOparse.GEOTypes.GSE'>


59

`.type()` returned `<class 'GEOparse.GEOTypes.GSE'>` confirming the correct object type, and `len(gse.metadata['sample_id'])` returned 59 confirming all samples were loaded. Further checks will be done to ensure samples' contents are correct. 

## 1.3 Sample Metadata Extraction & Validation

I check `GSM528831`'s, the first sample in the dataset, metadata to see where to extract information, then create a dataframe containing all samples with their diagnosis and illness duration to be used downstream (checking class balance, group comparisons, etc.). 

In [5]:
gse.gsms['GSM528831'].metadata

{'title': ['Control-1'],
 'geo_accession': ['GSM528831'],
 'status': ['Public on Mar 31 2010'],
 'submission_date': ['Mar 30 2010'],
 'last_update_date': ['Aug 12 2022'],
 'type': ['RNA'],
 'channel_count': ['1'],
 'source_name_ch1': ['prefrontal cortex'],
 'organism_ch1': ['Homo sapiens'],
 'taxid_ch1': ['9606'],
 'characteristics_ch1': ['brain region: BA46',
  'stage of illness [short doi=<5 yrs; intermediate doi=7-18yrs; long doi=>28 yrs]: short duration of illness - control',
  'Sex: M',
  'age: 38',
  'tissue ph: 6.42',
  'pmi (hrs): 46',
  'type of drug: NA',
  'drug dose (chlorpromazine equivalents): NA'],
 'molecule_ch1': ['total RNA'],
 'extract_protocol_ch1': ['Total RNA was extracted from the prefrontal cortex (Brodmann Area 46; 100 mg; left hemisphere) from all subjects as described previously (Desplats et al., 2006).  RNA quantification was determined by spectrophotometer readings, and quality by Agilent Bioanalyzer scans.  RNA integrity (RIN) numbers were not available at

Looking at `GSM528831`'s metadata, `characteristics_ch1` is the key that contains the information on diagnosis and duration of illness.

I will create a dataframe using `characteristics_ch1` to extract duration and diagnosis. 

I will do this by looping over all the samples (`gse.gsms.items()`) applying the following parsing logic to each sample to create a dictionary of sample : sampleID/duration/diagnosis: splitting `characteristics_ch1`'s second item (contains the diagnosis and duration), on `:` to get two strings - one containing the duration and diagnosis, the other containing contextual information. I will then split that list again on the second item by `-` to get a new list containing only duration information and diagnosis. Lastly, I set `diagnosis` to the split string's second item which has only the diagnosis, and set `duration` to the first item of that split, after stripping whitespace and splitting on ' ' to isolate just the duration. 

I put that dictionary into a dataframe and apply `.T` to transpose it so that each row is a sample and the columns are `['sample_id', 'duration', 'diagnosis']`. 

In [21]:
df = {}
for gsm_id, gsm_obj in gse.gsms.items():
   a = gsm_obj.metadata['characteristics_ch1'][1].split(':')
   b = a[1].split('-')
   duration = b[0].strip().split(' ')[0]
   diagnosis = b[1].strip()
   sample_id = gsm_id
   df[gsm_id] = [sample_id, duration, diagnosis]
   
gsm_df = pd.DataFrame(df).T
gsm_df.columns = ['sample_id', 'duration', 'diagnosis']
gsm_df.head()

,sample_id,duration,diagnosis
GSM528831,GSM528831,short,control
GSM528832,GSM528832,short,control
GSM528833,GSM528833,short,control
GSM528834,GSM528834,short,control
GSM528835,GSM528835,short,control


`.head()` of `gsm_df` confirms the dataframe has correactly labeld columns and rows are samples.

To further check the dataframe was constructed correctly, I will check a specific schizophrenia sample to ensure that those samples were loaded correctly, and the `.value_counts()` of `diagnosis` and `duration` to ensure the correct number of diagnoses and three duration categories are present. 

In [25]:
gsm_df.loc['GSM528885']

sample_id        GSM528885
duration              long
diagnosis    schizophrenia
Name: GSM528885, dtype: str

`GSM528885` shows `duration = long` and `diagnosis = schizophrenia` — confirming schizophrenia samples were parsed correctly.

In [24]:
print(gsm_df['diagnosis'].value_counts())
gsm_df['duration'].value_counts()

diagnosis
schizophrenia    30
control          29
Name: count, dtype: int64


duration
intermediate    28
long            16
short           15
Name: count, dtype: int64

`value_counts()` for both diagnosis and duration show the correct number of diagnoses (30 schizophrenia and 29 control — cont 7 was considered an outlier and excluded) and the three duration categories (intermediate: 28, long: 16, short: 15).

I will run a `pd.crosstab` on `diagnosis` and `duration` to check the spread of control and schizophrenia durations. 

In [26]:
pd.crosstab(gsm_df['diagnosis'], gsm_df['duration'])

duration,intermediate,long,short
diagnosis,,,
control,14,8,7
schizophrenia,14,8,8


The crosstab shows a proportional pattern (control 14/8/7 vs. schizophrenia 14/8/8) that is consistent with control being stage matched to schizophrenia subjects, even though the original study summary only explicitly states age/sex-matching. 

## 1.4 Expression Matrix Load & Inspect